In [1]:
import os
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
from sklearn.preprocessing import normalize
import google.generativeai as genai
from dotenv import load_dotenv

# ==============================================================================
# [1] 환경 설정 (경로 문제 해결 강화)
# ==============================================================================

# 1. 현재 작업 경로(C:\STUDY-DATA)가 아니라, 이 파일이 있는 폴더(.../12_25)를 기준으로 .env 찾기
current_path = Path.cwd() # 현재 터미널 경로

# 가능한 .env 경로 후보들
env_candidates = [
    Path(".env"),                                      # 1. 현재 폴더
    Path("third_week/12_25/.env"),                     # 2. 터미널이 상위 폴더일 때
    Path("data_csv").parent / ".env",                  # 3. 데이터 폴더 옆
    Path(__file__).parent / ".env" if "__file__" in locals() else None # 4. 스크립트 위치 (ipynb에선 동작 안 할 수 있음)
]

env_loaded = False
for env_path in env_candidates:
    if env_path and env_path.exists():
        load_dotenv(dotenv_path=env_path)
        print(f"✅ .env 파일을 찾았습니다: {env_path}")
        env_loaded = True
        break

if not env_loaded:
    print("⚠️ .env 파일을 자동으로 찾지 못했습니다. 시스템 환경변수를 확인합니다.")

# 2. API 키 가져오기
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

# 🚨 [비상 대책] 만약 여전히 키가 없다면, 아래 따옴표 안에 키를 직접 붙여넣으세요.
if not GOOGLE_API_KEY:
    # 예: GOOGLE_API_KEY = "AIzaSyD..." 
    # 여기에 본인의 키를 직접 입력하고 주석(#)을 푸세요.
    # GOOGLE_API_KEY = "여기에_구글_API_키를_붙여넣으세요" 
    pass

if not GOOGLE_API_KEY:
    raise ValueError(
        "\n❌ [오류] GOOGLE_API_KEY를 찾을 수 없습니다."
        "\n1. .env 파일이 현재 폴더나 'third_week/12_25' 안에 있는지 확인하세요."
        "\n2. .env 파일 안에 'GOOGLE_API_KEY=AIza...' 형태로 저장되어 있는지 확인하세요."
        "\n3. 정 안되면 위 코드의 [비상 대책] 부분에 키를 직접 입력하세요."
    )

genai.configure(api_key=GOOGLE_API_KEY)
EMBEDDING_MODEL = "models/text-embedding-004"
print("✅ Google API 설정 완료")

# ==============================================================================
# [2] 임베딩 친화적으로 강화된 톤 설명 데이터
# ==============================================================================
TONE_TEXTS = {
    "Scientific": """
    Scientific 톤은 객관적인 연구 데이터와 임상 실험 결과를 바탕으로 제품의 효능을 논리적으로 입증하는 신뢰 중심의 톤이다. 
    감정보다는 이성에 호소하며, '메커니즘', '특허 기술', '솔루션', '수치 검증', '성분 분석'과 같은 전문적인 어휘를 사용하여 
    제품의 확실한 효과와 기술적 우위를 명확하고 분석적으로 전달한다.
    """,
    
    "Emotional": """
    Emotional 톤은 고객의 일상과 감정에 깊이 공감하며 따뜻한 위로와 행복을 전하는 감성 스토리텔링 톤이다. 
    기능적 설명보다는 마음을 움직이는 데 집중하며, '소중한 당신', '기분 좋은 휴식', '사랑스러운', '함께하는 추억', '마음의 선물'과 같은 
    부드럽고 서정적인 어휘를 통해 브랜드와 고객 사이의 정서적 유대감을 형성한다.
    """,
    
    "Luxury": """
    Luxury 톤은 압도적인 품격과 희소성을 강조하여 브랜드의 프리미엄 가치를 전달하는 고품격 톤이다. 
    오랜 헤리티지와 장인정신, 그리고 남들과 다른 특별함을 부각한다. 
    '시간을 초월한 걸작', '고귀한 가치', '노블레스', '최상의 경험', '압도적 아우라' 등 
    무게감 있고 세련된 어휘를 통해 고객에게 프라이빗하고 특별한 대우를 받는다는 느낌을 선사한다.
    """,
    
    "Casual": """
    Casual 톤은 옆집 친구처럼 친근하고 솔직한 화법으로 부담 없이 소통하는 일상적이고 활기찬 톤이다. 
    복잡한 격식 대신 쉽고 간결한 구어체를 사용하며 유머와 재치를 곁들인다. 
    '진짜 꿀팁', '완전 강추', '가성비 대박', '솔직 후기', '그냥 써봐'처럼 
    트렌디하고 가벼운 표현을 통해 고객과 즉각적이고 가까운 관계를 맺는다.
    """
}

# ==============================================================================
# [3] 임베딩 함수
# ==============================================================================
def get_embedding_gemini(text):
    """문장 전체의 맥락을 반영한 임베딩 벡터 생성"""
    try:
        clean_text = text.replace('\n', ' ').strip()
        result = genai.embed_content(
            model=EMBEDDING_MODEL,
            content=clean_text,
            task_type="retrieval_document"
        )
        return np.array(result['embedding'])
    except Exception as e:
        print(f"❌ 임베딩 실패: {e}")
        return None

# ==============================================================================
# [4] 벡터 생성 및 저장 로직
# ==============================================================================
def build_tone_assets(tone_texts):
    tone_vectors = {}
    meta_rows = []

    print(f"🚀 Start processing {len(tone_texts)} tones (Rich Description Mode)...")

    for tone_label, description in tone_texts.items():
        print(f" -> Processing '{tone_label}'...", end="")
        
        vector = get_embedding_gemini(description)
        
        if vector is not None:
            # 정규화
            norm_vector = normalize(vector.reshape(1, -1), norm='l2')[0]
            tone_vectors[tone_label] = norm_vector
            
            meta_rows.append({
                "tone_id": tone_label,
                "description_preview": description.strip()[:50] + "...",
                "full_description": description.strip(),
                "model_used": EMBEDDING_MODEL
            })
            print(f" ✅ Complete. (Dim: {len(norm_vector)})")
        else:
            print(" ❌ Failed.")

    return tone_vectors, pd.DataFrame(meta_rows)

# ==============================================================================
# [5] 실행 메인
# ==============================================================================
if __name__ == "__main__":
    try:
        vectors_pkl, df_meta = build_tone_assets(TONE_TEXTS)
        
        with open("tone_vectors.pkl", "wb") as f:
            pickle.dump(vectors_pkl, f)
        print("\n📂 [Saved] 'tone_vectors.pkl' saved successfully.")
        
        df_meta.to_csv("tone_metadata.csv", index=False, encoding="utf-8-sig")
        print("📂 [Saved] 'tone_metadata.csv' saved successfully.")
        
        # [검증]
        print("\n🔍 [최종 검증]")
        if 'Scientific' in vectors_pkl and 'Luxury' in vectors_pkl:
            vec_sci = vectors_pkl['Scientific']
            vec_lux = vectors_pkl['Luxury']
            similarity = np.dot(vec_sci, vec_lux)
            
            print(f"Scientific vs Luxury 유사도: {similarity:.4f}")
            if similarity < 0.99:
                print("✅ 두 벡터가 확연히 구분됩니다. 성공!")
            else:
                print("🚨 벡터가 너무 유사합니다.")
        else:
            print("⚠️ 일부 톤의 벡터가 생성되지 않았습니다.")

    except Exception as e:
        print(f"\n❌ 오류 발생: {e}")

c:\STUDY-DATA\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\박윤후\AppData\Local\Temp\ipykernel_16148\1666876863.py:7: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


✅ .env 파일을 찾았습니다: .env
✅ Google API 설정 완료
🚀 Start processing 4 tones (Rich Description Mode)...
 -> Processing 'Scientific'... ✅ Complete. (Dim: 768)
 -> Processing 'Emotional'... ✅ Complete. (Dim: 768)
 -> Processing 'Luxury'... ✅ Complete. (Dim: 768)
 -> Processing 'Casual'... ✅ Complete. (Dim: 768)

📂 [Saved] 'tone_vectors.pkl' saved successfully.
📂 [Saved] 'tone_metadata.csv' saved successfully.

🔍 [최종 검증]
Scientific vs Luxury 유사도: 0.7942
✅ 두 벡터가 확연히 구분됩니다. 성공!
